# XGBoost Starter Notebook - LB Ensemble 0.933! Wow!
In this notebook, we train an XGBoost model and ensemble it with the best public notebook. The best public notebook achieves `LB = 0.915` and our ensemble achieves `LB = 0.935` Wow!

# UPDATE
In version 2 we increase `max_depth` from `3` to `6` to allow for more feature interaction and we add regularation `alpha=1` to prevent overifitting which improves generalization to LB test data. This improves ensemble LB score `LB 0.926 => LB 0.935` woohoo!

# Load Data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv("/kaggle/input/playground-series-s5e3/train.csv")
print("Train shape", train.shape )
train.head()

In [ ]:
test = pd.read_csv("/kaggle/input/playground-series-s5e3/test.csv")
print("Test shape:", test.shape )
test.head()

In [ ]:
RMV = ['rainfall','id']
FEATURES = [c for c in train.columns if not c in RMV]
print("Our features are:")
print( FEATURES )

# XGBoost
We train 5 fold XGBoost model. We use `max_depth=6`, `colsample_bytree=0.9`, and `subsample=0.9`. These are the 3 hyperparameters that i like to tune.

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor, XGBClassifier
import xgboost
print("Using XGBoost version",xgboost.__version__)

In [ ]:
%%time
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
    
oof_xgb = np.zeros(len(train))
pred_xgb = np.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print("#"*25)
    print(f"### Fold {i+1}")
    print("#"*25)
    
    x_train = train.loc[train_index,FEATURES].copy()
    y_train = train.loc[train_index,"rainfall"]    
    x_valid = train.loc[test_index,FEATURES].copy()
    y_valid = train.loc[test_index,"rainfall"]
    x_test = test[FEATURES].copy()

    model = XGBClassifier(
        device="cuda",
        max_depth=6,  
        colsample_bytree=0.9, 
        subsample=0.9, 
        n_estimators=10_000,  
        learning_rate=0.1, 
        eval_metric="auc",
        early_stopping_rounds=100,
        alpha=1,
    )
    model.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],  
        verbose=100 
    )

    # INFER OOF
    oof_xgb[test_index] = model.predict_proba(x_valid)[:,1]
    # INFER TEST
    pred_xgb += model.predict_proba(x_test)[:,1]

# COMPUTE AVERAGE TEST PREDS
pred_xgb /= FOLDS

In [ ]:
from sklearn.metrics import roc_auc_score
true = train.rainfall.values
m = roc_auc_score(true, oof_xgb)
print(f"XGBoost CV Score AUC = {m:.3f}")

In [ ]:
feature_importance = model.feature_importances_
importance_df = pd.DataFrame({
    "Feature": FEATURES,  
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)
plt.figure(figsize=(10, 5))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.gca().invert_yaxis()  
plt.show()

# Submission CSV Ensemble!
We load the best public notebook from version 17 of public notebook which achieves `LB 0.915` (from [here][1]). Then we ensemble our new XGBoost model preditions with weights `-1.0 * XGB + 2.0 * Public`. We use `scipy.stats.rankdata` to normalize predictions before ensemble. We achieve `LB 0.935` hooray!

[1]: https://www.kaggle.com/code/act18l/auc-post-processing?scriptVersionId=227652340

In [ ]:
print("Best Public Notebook achieves LB = 0.915!")
best_public = pd.read_csv("/kaggle/input/lb-915-public-notebook/submission.csv")
display( best_public.head() )
best_public = best_public.rainfall.values

In [ ]:
from scipy.stats import rankdata

print("Ensemble achieves LB = 0.935! Hooray!")
sub = pd.read_csv("/kaggle/input/playground-series-s5e3/sample_submission.csv")
sub.rainfall = -1 * rankdata( pred_xgb ) + 2 * rankdata( best_public )
sub.rainfall = rankdata( sub.rainfall ) / len(sub)
print( sub.shape )
sub.to_csv(f"submission_ensemble.csv",index=False)
sub.head()